# EXTRACT FEATURES FROM A DATASET OF URLS

## INTRODUCTION
This document focuses on extracting features from urls and standardizes the features


## Step 1: read dataset
- using read_csv from pandas to read dataset

In [16]:
import pandas as pd
file_path = r'G:\url-analysis\src\data\malicious_phish.csv'
url_data = pd.read_csv(file_path, encoding_errors = 'ignore')

In [17]:
url_data.columns

Index(['url', 'type'], dtype='str')

In [18]:
print(url_data.head())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement


In [19]:
print(url_data.shape)

(653490, 2)


In [20]:
url_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 653490 entries, 0 to 653489
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   url     653490 non-null  str  
 1   type    653490 non-null  str  
dtypes: str(2)
memory usage: 51.7 MB


## Step 2: create a new csv file
- create a file path and a new csv file

In [21]:
import os
new_file_path = r'G:\url-analysis\src\data\malicious_phish_processed.csv'

folder_path = os.path.dirname(new_file_path)
os.makedirs(folder_path, exist_ok=True)
if not os.path.exists(new_file_path):
    url_data.to_csv(new_file_path, index=False)


## Step 3: update the new csv file
- Copy data from dataset and add a new column url length to malicious_phish_processed.csv

In [24]:
%load_ext autoreload
%autoreload 2
import src.features.feature_extraction as fex

file_path_processed = r'G:\url-analysis\src\data\malicious_phish_processed.csv'
processed_data = pd.read_csv(file_path_processed, encoding_errors = 'ignore')
processed_data['url length'] = processed_data['url'].apply(fex.url_length)
processed_data.to_csv(file_path_processed, index=False)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 4: update features to the dataset

In [25]:
%%time
whitelist_path = r'G:\url-analysis\src\data\domain.txt'
#load and process whitelist
preprocessed_list = fex.load_and_preprocess_whitelist(whitelist_path)

#create feature column names
feature_cols = [
    'number_of_part', 'has_scheme', 'has_netloc', 'has_path', 'has_params', 'has_query', 'has_fragment',
    'has_username', 'has_password', 'has_port', 'has_subdomain', 'has_domain', 'has_suffix',
    'netloc_length', 'path_length', 'query_length', 'fragment_length', 'subdomain_length', 'domain_length',
    'url_entropy', 'netloc_entropy', 'path_entropy', 'query_entropy', 'subdomain_entropy', 'domain_entropy',
    'number_of_subdomain', 'hyphen_in_subdomain', 'hyphen_in_domain', 'unicode', 'punycode', 'at_sign_in_netloc',
    'slash_in_path', 'dot_in_path', 'strange_in_query', 'equal_in_query', 'ampersand_in_query',
    'normalized_levenshtein_domain', 'normalized_levenshtein_subdomain', 'random_domain_check', 'random_subdomain_check',
    'number_ratio_domain', 'number_ratio_subdomain', 'repeated_domain_check', 'repeated_path_check', 'repeated_url_check',
    'longest_repeated_chain', 'ip_domain', 'suspicious_key_domain', 'suspicious_key_subdomain', 'suspicious_key_path',
    'suspicious_key_query', 'shortened', 'has_uuid_path', 'download_param', 'free_host', 'free_host_download', 'suspicious suffix'
]

#extract features from url columns
extracted_data = processed_data['url'].apply(lambda url : fex.features_extraction(url, preprocessed_list)).tolist()

#add features to RAM
processed_data[feature_cols] = pd.DataFrame(extracted_data, index = processed_data.index)

#save features to csv file
file_path_processed = r'G:\url-analysis\src\data\malicious_phish_processed.csv'
processed_data.to_csv(file_path_processed, index = False)

CPU times: total: 59.6 s
Wall time: 60 s


## Step 5: Use one-hot encoding to classify scheme

In [26]:
from sklearn.preprocessing import OneHotEncoder

#Define 5 categories, use 2D array
categories = [['https', 'http', 'ftp', 'none', 'other']]

#extract scheme from each url
extracted_scheme = processed_data['url'].apply(fex.get_scheme)

#convert list to DataFrame
extracted_scheme_2D = extracted_scheme.to_frame()

#Create Encoder
encoder = OneHotEncoder(categories=categories, sparse_output=False,handle_unknown='ignore')

#use OneHotEncoder.fit_transform to transform categorized data to binary form after being fitted
encoder_array = encoder.fit_transform(extracted_scheme_2D)

#create column name
col_names = [f'is_{scheme}' for scheme in categories[0]]

#
df_encoded = pd.DataFrame(encoder_array, columns=col_names, index=processed_data.index)

#concat new features to DataFrame
processed_data = pd.concat([processed_data, df_encoded], axis = 1)

#save to csv
processed_data.to_csv(file_path_processed, index = False)


(653490, 5)


## Step 6: Standardized the data using Min-Max Scaling and Z-score Standardization